### If you need help with getting all the rows of data, use this as a reference

In [82]:
import polars as pl
from supabase import create_client
from dotenv import load_dotenv
import os

In [83]:
COLUMNS = ["id", "created_at", "source_id", "source_type", "permit_id", "permit_type", 
           "kind", "company_name", "kilowatt_value", "issue_date", "apply_date", "latitude", "longitude", 
           "full_address", "city", "state", "county", "postal_code", "is_active", "is_system_size_estimation"]

In [84]:
load_dotenv(override=True)

True

In [85]:


url = os.getenv("DATABASE_SEND_URL")
key = os.getenv("DATABASE_SEND_KEY")

client = create_client(url, key)

In [86]:
def fetch_all(table: str) -> list[dict]:
    rows, offset, page_size = [], 0, 1000
    while True:
        result = (
            client.table(table)
            .select(", ".join(COLUMNS))
            .range(offset, offset + page_size - 1)
            .execute()
        )
        rows.extend(result.data)
        if len(result.data) < page_size:
            break
        offset += page_size
    return rows

print("Fetching..")
rows = fetch_all("records")

df = pl.DataFrame(rows)


Fetching..


In [87]:
print(df.shape)      # (37000, n_cols)
print(df.head())
print(df.dtypes)

(37901, 20)
shape: (5, 20)
┌────────────┬────────────┬───────────┬───────────┬───┬────────┬───────────┬───────────┬───────────┐
│ id         ┆ created_at ┆ source_id ┆ source_ty ┆ … ┆ county ┆ postal_co ┆ is_active ┆ is_system │
│ ---        ┆ ---        ┆ ---       ┆ pe        ┆   ┆ ---    ┆ de        ┆ ---       ┆ _size_est │
│ str        ┆ str        ┆ str       ┆ ---       ┆   ┆ str    ┆ ---       ┆ bool      ┆ imation   │
│            ┆            ┆           ┆ str       ┆   ┆        ┆ str       ┆           ┆ ---       │
│            ┆            ┆           ┆           ┆   ┆        ┆           ┆           ┆ bool      │
╞════════════╪════════════╪═══════════╪═══════════╪═══╪════════╪═══════════╪═══════════╪═══════════╡
│ 4ec116e4-a ┆ 2026-03-17 ┆ 8004cab0- ┆ SITE      ┆ … ┆ null   ┆ 92835     ┆ true      ┆ false     │
│ 980-4c87-9 ┆ T08:06:38. ┆ e823-4a54 ┆           ┆   ┆        ┆           ┆           ┆           │
│ ae7-8f4e7e ┆ 438358+00: ┆ -9cb9-231 ┆           ┆   ┆        ┆